In [1]:
!pip install feature-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 6.6 MB/s eta 0:00:00


In [2]:
from sklearn.base import (BaseEstimator, TransformerMixin, clone)
from sklearn.utils.validation import check_is_fitted
from sklearn.compose import ColumnTransformer
from feature_engine.datetime import DatetimeFeatures
from feature_engine.outliers import ArbitraryOutlierCapper
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

Scikit-learn's Documentation tells me that all parameters should be stored exactly and all logic should be performed in transform, unless some pre-processing is neededl; which is done in fit.

All the transformers are designed with this philosophy in mind.

In [3]:
class WeatherAnomalyCleaner(BaseEstimator, TransformerMixin):

    DEFAULT_LIMITS = {
        "Temperature(F)": (-60, 130),
        "Humidity(%)": (1, 100),
        "Pressure(in)": (20, 32.5),
        "Visibility(mi)": (0, 30),
        "Wind_Speed(mph)": (0, 150),
    }
    
    @staticmethod
    def _validate(weather_limits):
        for col, (low,high) in weather_limits.items():
            if low > high:
                raise ValueError(f"Lower Bound > Upper Bound for Key - {col}")
    
    def __init__(self,*,
                 weather_limits = None,
                 drop = True,
                 copy = False,
                 missing_values='ignore'):
        '''
        DOCSTRING

        Pretty simple transformer, drops all data that is not within the range.
        We may choose to cap data instead by setting 'drop' to False.

        'missing_values' is used only if we are capping. Ignores by default, the user may change to Raise.

        Use np.inf or -np.inf to indicate that no upper or lower bound should be applied.

        NOTE: Missing Values are untouched if we drop the Anomalous rows

        PARAMETERS:
            > weather_limits = A dictionary of form {column_name:(min,max),}
            > drop = A boolean specifying if we drop Anomalous rows or cap the data instead
            > copy = Whether to return a copy of the dataframe or perform changes in-place
            > missing_values = Used if we cap values, 'ignore' by default. May be changed to 'raise'
        '''        
        self.weather_limits = weather_limits
        self.drop = drop
        self.copy = copy
        self.missing_values = missing_values

    def fit(self, X, y=None):
        self._limits = (
            self.DEFAULT_LIMITS 
            if self.weather_limits is None
            else self.weather_limits
        )

        type(self)._validate(self._limits)
        return self

    def transform(self, X):
        check_is_fitted(self,"_limits")
        
        limits = self._limits
        
        if self.copy: X = X.copy()

        invalid = np.zeros(len(X), dtype=bool)

        if self.drop:
            for column, (low, high) in limits.items():
    
                invalid |= (
                    X[column].notna()
                    &
                    ~X[column].between(low, high)
                )
    
            return X.loc[~invalid].reset_index(drop=True)
        
    
        capper = ArbitraryOutlierCapper(
            max_capping_dict = {col:high for col, (_,high) in limits.items()},
            min_capping_dict = {col:low for col, (low,_) in limits.items()},
            missing_values = self.missing_values
        )

        return capper.fit_transform(X)

In [4]:
class DateTimeFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self,*,
                copy = False):
        '''DOCSTRING

        This Transformer is to convert DateTimeFeatures into a more useful form.

        PARAMETERS:
            > copy = Whether to return a copy of the dataframe or perform changes in-place
        '''
        self.copy = copy

    def fit(self, X, y=None):
        ...

    def transform(self, X):
        if self.copy: X = X.copy()